# Stock Price Prediction
Predicting stock closing prices using historical market data.

## Project Overview
Regression project using Yahoo Finance data to predict stock closing prices with boosting models.

## Learning Objectives
- Download financial data programmatically
- Engineer lag and rolling features for stock data
- Compare ML models for price prediction

## Problem Statement
Predict the next-day closing price of a stock given historical OHLCV data.

## Why This Project Matters
Stock prediction is a classic ML problem that teaches feature engineering, overfitting risks, and the efficient market hypothesis.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | Yahoo Finance (yfinance) |
| **Ticker** | AAPL |
| **Target** | Close price |
| **Period** | 10 years |

## Dataset Source & License
Yahoo Finance public market data. Free for personal/educational use.

## Environment Setup

In [1]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
try: import yfinance
except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','yfinance','-q'])

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Imports

In [2]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
import yfinance as yf
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [3]:
TARGET = 'Close'
TICKER = 'AAPL'
PERIOD = '10y'

## Dataset Loading

In [4]:
df = yf.download(TICKER, period=PERIOD, auto_adjust=True).reset_index()
# Flatten MultiIndex columns if present
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0] if c[1] == '' or c[1] == TICKER else c[0] for c in df.columns]
print(f'Shape: {df.shape}')
df.head()

[*********************100%***********************]  1 of 1 completed

Shape: (2514, 6)


,Date,Close,High,Low,Open,Volume
0,2016-04-13,25.374870,25.442813,25.094035,25.094035,133029200
1,2016-04-14,25.388464,25.454144,25.214075,25.279754,101895600
2,2016-04-15,24.878880,25.433758,24.851703,25.390726,187756000
3,2016-04-18,24.342121,24.675046,24.219821,24.661458,243286000
4,2016-04-19,24.213024,24.459887,24.059017,24.432709,129539600


## Data Validation

In [5]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Date range: {df["Date"].min()} to {df["Date"].max()}')
print(df[TARGET].describe())

Missing: 0
Duplicates: 0
Date range: 2016-04-13 00:00:00 to 2026-04-13 00:00:00
count    2514.000000
mean      120.691378
std        75.385949
min        20.584818
25%        44.162038
50%       127.161327
75%       176.540710
max       285.922455
Name: Close, dtype: float64


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0,0].plot(df['Date'], df[TARGET])
axes[0,0].set_title(f'{TICKER} Close Price')
axes[0,1].plot(df['Date'], df['Volume'])
axes[0,1].set_title('Volume')
axes[1,0].hist(df[TARGET].pct_change().dropna(), bins=100, edgecolor='black')
axes[1,0].set_title('Daily Returns Distribution')
corr = df.select_dtypes('number').corr()
im = axes[1,1].imshow(corr, cmap='coolwarm', aspect='auto')
axes[1,1].set_title('Correlation')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [7]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

count    2514.000000
mean      120.691378
std        75.385949
min        20.584818
25%        44.162038
50%       127.161327
75%       176.540710
max       285.922455
Name: Close, dtype: float64

Skewness: 0.28


## Train/Test Split

In [8]:
# Feature Engineering: lag and rolling features
df = df.sort_values('Date').reset_index(drop=True)
for lag in [1, 2, 3, 5, 10]:
    df[f'Close_lag{lag}'] = df[TARGET].shift(lag)
for w in [5, 10, 20]:
    df[f'Close_ma{w}'] = df[TARGET].rolling(w).mean()
    df[f'Close_std{w}'] = df[TARGET].rolling(w).std()
df['Returns'] = df[TARGET].pct_change()
df['Volume_ma5'] = df['Volume'].rolling(5).mean()

# Drop Date and NaN rows from lagging
df = df.drop(columns=['Date']).dropna().reset_index(drop=True)

X = df.drop(columns=[TARGET])
y = df[TARGET]
# Time-aware split: last 20% as test
split_idx = int(len(X) * (1 - TEST_SIZE))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (1996, 17), Test: (499, 17)


## Preprocessing
Created lag features and rolling statistics. Used time-ordered split to prevent data leakage.

## Feature Engineering
Lag features (1-10 days), moving averages (5/10/20 day), rolling std, daily returns, volume MA.

## Baseline Model

In [9]:
bl = LinearRegression()
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline LR: R2={r2_score(y_test, bl_pred):.4f}  RMSE={root_mean_squared_error(y_test, bl_pred):.4f}  MAE={mean_absolute_error(y_test, bl_pred):.4f}')

Baseline LR: R2=0.9985  RMSE=1.0475  MAE=0.7814


## LazyPredict Benchmark

In [10]:
# --- LazyPredict Benchmark ---
try:
    from lazypredict.Supervised import LazyRegressor
    n_lp = min(5000, len(X_train))
    lr = LazyRegressor(verbose=0, ignore_warnings=True)
    lp_models, _ = lr.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

                             Adjusted R-Squared  R-Squared      RMSE  \
Model                                                                  
Lars                                   0.998432   0.998485  1.046010   
LarsCV                                 0.998432   0.998485  1.046030   
LassoLarsCV                            0.998427   0.998481  1.047530   
LinearRegression                       0.998427   0.998481  1.047530   
TransformedTargetRegressor             0.998427   0.998481  1.047530   
RANSACRegressor                        0.998427   0.998481  1.047530   
LassoLarsIC                            0.998426   0.998480  1.047739   
BayesianRidge                          0.998425   0.998479  1.048262   
RidgeCV                                0.998097   0.998162  1.152144   
HuberRegressor                         0.997569   0.997652  1.302216   
Ridge                                  0.996581   0.996698  1.544308   
PassiveAggressiveRegressor             0.996159   0.996290  1.63

## FLAML AutoML

In [11]:
# --- FLAML AutoML ---
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='regression', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  R2={r2_score(y_test, flaml_pred):.4f}  RMSE={root_mean_squared_error(y_test, flaml_pred):.4f}  MAE={mean_absolute_error(y_test, flaml_pred):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

FLAML skipped: XGBModel.fit() got an unexpected keyword argument 'callbacks'


## Additional Models: CatBoost, LightGBM, XGBoost

In [12]:
# --- Boosting Models ---
models = {}
# CatBoost
cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

# LightGBM
lgb = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

# XGBoost
xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0)
xgb.fit(X_train, y_train)
models['XGBoost'] = xgb

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: R2={r2_score(y_test, p):.4f}  RMSE={root_mean_squared_error(y_test, p):.4f}  MAE={mean_absolute_error(y_test, p):.4f}")

CatBoost: R2=-1.8853  RMSE=45.6510  MAE=38.4561
LightGBM: R2=-1.8097  RMSE=45.0490  MAE=37.5862
XGBoost: R2=-1.7642  RMSE=44.6832  MAE=37.1994


## Top 3 Model Selection

In [13]:
# --- Top 3 Selection ---
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {'R2': r2_score(y_test, p), 'RMSE': root_mean_squared_error(y_test, p), 'MAE': mean_absolute_error(y_test, p)}

results_df = pd.DataFrame(all_results).T.sort_values('R2', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

                R2       RMSE        MAE
XGBoost  -1.764236  44.683235  37.199406
LightGBM -1.809672  45.048969  37.586187
CatBoost -1.885268  45.650987  38.456120

Top 3: ['XGBoost', 'LightGBM', 'CatBoost']


## Final Evaluation of Top 3

In [14]:
# --- Final Evaluation of Top 3 ---
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    r2 = r2_score(y_test, p)
    rmse = root_mean_squared_error(y_test, p)
    mae = mean_absolute_error(y_test, p)
    final_results[name] = {'R2': r2, 'RMSE': rmse, 'MAE': mae}
    print(f"{name}: R2={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
names = list(final_results.keys())
for i, metric in enumerate(['R2', 'RMSE', 'MAE']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

XGBoost: R2=-1.7642  RMSE=44.6832  MAE=37.1994
LightGBM: R2=-1.8097  RMSE=45.0490  MAE=37.5862
CatBoost: R2=-1.8853  RMSE=45.6510  MAE=38.4561


## Error Analysis

In [15]:
# --- Error Analysis ---
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)
residuals = y_test - preds

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(residuals, bins=50, edgecolor='black')
axes[0].set_title('Residual Distribution')
axes[0].axvline(0, color='red', linestyle='--')
axes[1].scatter(preds, residuals, alpha=0.3, s=5)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals vs Predicted')
axes[2].scatter(y_test, preds, alpha=0.3, s=5)
mn, mx = y_test.min(), y_test.max()
axes[2].plot([mn, mx], [mn, mx], 'r--')
axes[2].set_title('Actual vs Predicted')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## Interpretation & Business Insight
Lag features dominate predictions — the best predictor of tomorrow's price is today's price. This reflects market efficiency.

## Limitations
- Using lag of target as feature means model mostly learns 'price stays similar'
- No external signals (news, earnings)
- Doesn't account for market regime changes

## How to Improve
- Add sentiment features from news
- Use time-series models (LSTM, Transformer)
- Include technical indicators (RSI, MACD, Bollinger Bands)

## Production Considerations
- Real-time data feed required
- Walk-forward validation for backtesting
- Risk management layer needed

## Common Mistakes
- Using future data in features (look-ahead bias)
- Random train/test split instead of time-ordered
- Confusing high R2 with trading profitability

## Mini Challenge
1. Try predicting returns instead of prices
2. Add RSI and MACD indicators
3. Use walk-forward validation

## Final Summary
Stock price prediction using lag features and boosting models. High R2 is expected due to autocorrelation — does not imply trading alpha.

In [16]:
# --- Save Metrics ---
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))

Saved metrics.json
{
  "XGBoost": {
    "R2": -1.7642,
    "RMSE": 44.6832,
    "MAE": 37.1994
  },
  "LightGBM": {
    "R2": -1.8097,
    "RMSE": 45.049,
    "MAE": 37.5862
  },
  "CatBoost": {
    "R2": -1.8853,
    "RMSE": 45.651,
    "MAE": 38.4561
  },
  "best_model": "XGBoost"
}
